In [1]:
import numpy as np
import pandas as pd
import re
import nltk
import spacy
import string

In [2]:
df=pd.read_csv("development.csv",delimiter=",", index_col="Id")

### *Source* feature inspection

In [3]:
n_nan_source = df['source'].isna().sum()
n_empty_soruce = df['source'].astype(str).str.strip().eq('').sum()
print(f"Number of NaN rows: {n_nan_source}")
print(f"Number of empty rows: {n_empty_soruce}")
source_counts = df['source'].value_counts()
selected_sources = source_counts[source_counts >= 50].index
print(f"Relevant Sources:\n{selected_sources}")
coverage = source_counts[selected_sources].sum() / len(df)
print(f"Number of selected sources: {len(selected_sources)}")
print(f"Percentage of selected sources: {coverage:.2%}")

Number of NaN rows: 0
Number of empty rows: 0
Relevant Sources:
Index(['Yahoo', 'Reuters', 'BBC', 'New', 'Washington', 'RedNova', 'Boston',
       'CNN', 'CNET', 'Topix.Net', 'Guardian', 'Motley', 'Register',
       'International', 'Forbes', 'Time', 'ABC', 'InfoWorld', 'San', 'Wired',
       'Xinhua', 'Computerworld', 'News', 'CSMonitor', 'PCWorld', 'Bloomberg',
       'Seattle', 'Ananova', '\N', 'Syfy.com', 'Voice', 'USA', 'Independent',
       'Scotsman', 'CBS', 'Rediff', 'Times', 'Channel', 'CBC', 'Newsday',
       'Newsweek', 'Houston', 'Australian', 'Daily', 'Telegraph.co.uk', 'ESPN',
       'Canada.com', 'BCC', 'Sports', 'Search', 'Chicago', 'Turkish', 'CNN/SI',
       'MSNBC', 'London', 'National', 'Financial', 'Toronto', 'Indianapolis',
       'Melbourne', 'Christian', 'Detroit', 'ZDNet.com', 'CTV', 'PC', 'ic',
       'NEWS.com.au', 'RTE', 'Scotland', 'Hindustan', 'NPR', 'Al-Jazeera',
       'Information', 'IPS', 'TechNewsWorld', 'News24', 'sportinglife.com',
       'Arizona',

### *Title* feature inspection

In [28]:
n_nan_title = df['title'].isna().sum()
n_empty_title = df['title'].astype(str).str.strip().eq('').sum()
n_placeholders_title = df['title'].astype(str).str.strip().eq('\\N').sum()
print(f"Number of NaN rows: {n_nan_title}")
print(f"Number of empty rows: {n_empty_title}")
print(f"Number of placeholders (\\N): {n_placeholders_title}")
print("Titles Sample:")
print(df['title'].sample(10,random_state=42))

Number of NaN rows: 1
Number of empty rows: 2
Number of placeholders (\N): 0
Titles Sample:
Id
56722                      Care charity suspends Iraq work
60844    Rice: No memory of CIA warning of attack \\n  ...
74780                Alicia Markova, noted ballerina, dies
52281    Jackson&#39;s &#39;King Kong&#39; to more batt...
54595                    Team Wants to Clone Human Embryos
8215                       India budget focuses on farming
56359    Psychologist says war harmed Marine \\n    (AP...
43072              Warner&#39;s Mistakes Costly for Giants
66533    New wireless Internet service set to leave its...
58831                      Eat what you want, but exercise
Name: title, dtype: object


### *Article* feature inspection

In [27]:
n_nan_article = df['article'].isna().sum()
n_empty_article = df['article'].astype(str).str.strip().eq('').sum()
n_placeholders_article = df['article'].astype(str).str.strip().eq('\\N').sum()
print(f"Number of NaN rows: {n_nan_article}")
print(f"Number of empty rows: {n_empty_article}")
print(f"Number of placeholders (\\N): {n_placeholders_article}")
print("Articles Sample")
print(df['article'].sample(10,random_state=42))

Number of NaN rows: 1
Number of empty rows: 7
Number of placeholders (\N): 1874
Articles Sample
Id
56722    Aid agency Care International has suspended it...
60844    <p><a href="http://us.rd.yahoo.com/dailynews/r...
74780    Alicia Markova, Britain's first great ballerin...
52281    Even as the cameras are ready to roll for Pete...
54595    Harvard scientists ask the school's ethics boa...
8215     India targets agriculture in its annual budget...
56359    <p><a href="http://us.rd.yahoo.com/dailynews/r...
43072    Giants quarterback Kurt Warner walked off the ...
66533    <p><a href="http://us.rd.yahoo.com/dailynews/r...
58831                                                   \N
Name: article, dtype: object


### *PageRank* feature inspection

In [6]:
n_nan_pr = df['page_rank'].isna().sum()
n_empty_pr = df['page_rank'].astype(str).str.strip().eq('').sum()
print(f"Number of NaN rows: {n_nan_pr}")
print(f"Number of empty rows: {n_empty_pr}")
print(df['page_rank'].sample(20,random_state=42))

Number of NaN rows: 0
Number of empty rows: 0
Id
56722    5
60844    5
74780    5
52281    5
54595    5
8215     5
56359    5
43072    5
66533    5
58831    5
56721    5
833      4
51498    5
54365    5
25540    5
36505    5
42401    5
52590    5
38271    5
28275    5
Name: page_rank, dtype: int64


In [7]:
n_nan_time = df['timestamp'].isna().sum()
n_empty_time = df['timestamp'].astype(str).str.strip().eq('').sum()
print(f"Number of NaN rows: {n_nan_time}")
print(f"Number of empty rows: {n_empty_time}")
print(df['timestamp'].sample(20,random_state=42))

Number of NaN rows: 0
Number of empty rows: 0
Id
56722    0000-00-00 00:00:00
60844    2006-10-02 23:30:43
74780    0000-00-00 00:00:00
52281    0000-00-00 00:00:00
54595    0000-00-00 00:00:00
8215     2007-02-28 10:21:13
56359    2007-02-17 04:34:44
43072    0000-00-00 00:00:00
66533    2007-02-14 22:03:40
58831    0000-00-00 00:00:00
56721    2007-12-14 02:33:56
833      2007-10-26 21:43:52
51498    2007-08-30 16:14:12
54365    2004-09-30 19:04:36
25540    2007-05-30 00:05:58
36505    2004-10-13 21:27:40
42401    2007-07-09 14:10:57
52590    2007-10-23 21:08:02
38271    0000-00-00 00:00:00
28275    0000-00-00 00:00:00
Name: timestamp, dtype: object


In [8]:
time_val=df['timestamp'].values
np.array([time_val=="0000-00-00 00:00:00"]).sum()

27750

In [9]:
def clean_sentences(text):
    if pd.isna(text) or text == "":
        return ""

df['title_clean'] = df['title'].apply(clean_text_feature)

NameError: name 'clean_text_feature' is not defined